In [4]:
from pathlib import Path
import os
import glob
import re
import numpy as np
import pandas as pd

# プロジェクトルートの特定 (カレントが root でも scripts/ でも対応)
current_path = Path.cwd()
PROJECT_ROOT = current_path.parent if (current_path / "benchmarks").exists() is False and (current_path.parent / "benchmarks").exists() else current_path
P1_DIR = PROJECT_ROOT / "benchmarks" / "P1_northwind"

print(f"プロジェクトルート: {PROJECT_ROOT}")
print(f"P1 音声確認先: {P1_DIR}")

# MMS_FA エンジンおよび評価関数のインポート
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from test_rhythm_align import align_word_characters, map_words_to_phonemes
from evaluate_benchmarks import compute_robust_metrics

PASSAGE_TEXT = (
    "The North Wind and the Sun were disputing which was the stronger, "
    "when a traveler came along wrapped in a warm cloak. "
    "They agreed that the one who first succeeded in making the traveler "
    "take his cloak off should be considered stronger than the other."
)

FUNCTION_WORDS = {
    "a", "an", "the", "in", "on", "at", "to", "for", "of", "from", "with", "by", "about",
    "i", "you", "he", "she", "it", "we", "they", "me", "him", "her", "us", "them", "his",
    "this", "that", "these", "those", "which", "what", "when", "who", "whom", "whose",
    "am", "is", "are", "was", "were", "be", "been", "have", "has", "had",
    "and", "but", "or", "so", "than", "as", "should", "off", "along"
}
VOWEL_SET = {"AA", "AE", "AH", "AO", "AW", "AY", "EH", "ER", "EY", "IH", "IY", "OW", "OY", "UH", "UW"}

def compute_rcf(w_aligns, phones):
    content_durs, func_durs = [], []
    for w in w_aligns:
        clean_w = re.sub(r"[^a-zA-Z']", "", w.get("word", "")).lower()
        if not clean_w:
            continue
        is_func = clean_w in FUNCTION_WORDS
        w_start, w_end = w.get("start", 0.0), w.get("end", 0.0)
        v_durs = [
            (p["end"] - p["start"]) * 1000.0 for p in phones
            if p.get("start", 0.0) >= w_start - 0.01 and p.get("end", 0.0) <= w_end + 0.01
            and re.sub(r"[0-9]", "", p.get("phone", "")) in VOWEL_SET
        ]
        if v_durs:
            (func_durs if is_func else content_durs).extend(v_durs)
    mean_c = np.mean(content_durs) if content_durs else 120.0
    mean_f = np.mean(func_durs) if func_durs else 50.0
    return round(mean_c / max(mean_f, 15.0), 2)

cohorts = ["US-Native", "UK-Native", "JP-Accented", "JP-Mora-Forced"]
records = []

print("P1 (The North Wind and the Sun) 音声を解析中...")
for cohort in cohorts:
    wavs = glob.glob(str(P1_DIR / cohort / "*.wav"))
    print(f"  [{cohort}]: {len(wavs)} ファイル検出")
    for w_file in wavs:
        try:
            w_aligns = align_word_characters(w_file, PASSAGE_TEXT)
            phones = map_words_to_phonemes(w_aligns)
            m = compute_robust_metrics(phones)
            rcf = compute_rcf(w_aligns, phones)
            records.append({
                "cohort": cohort,
                "file": os.path.basename(w_file),
                "pct_v": m["pct_v"],
                "schwa_p75": m["schwa_p75_ms"],
                "npvi": m["npvi_clipped"],
                "rcf": rcf
            })
        except Exception as e:
            print(f"Error {w_file}: {e}")

df = pd.DataFrame(records)
out_csv = P1_DIR / "northwind_benchmark_results.csv"
df.to_csv(out_csv, index=False)
print(f"\n✅ 解析完了: {len(df)} サンプルを保存 -> {out_csv}")

print("\n" + "="*70)
print("📊 コーホート別 指標サマリー (平均 ± 標準偏差)")
print("="*70)
summary = df.groupby("cohort")[["pct_v", "schwa_p75", "rcf", "npvi"]].agg(
    lambda x: f"{np.mean(x):.1f} ± {np.std(x):.1f}"
)
print(summary.to_string())

# ネイティブ群（US + UK）の mu と cov
native_df = df[df["cohort"].isin(["US-Native", "UK-Native"])]
pts = native_df[["pct_v", "schwa_p75"]].values
mu = np.mean(pts, axis=0)
cov = np.cov(pts, rowvar=False)

print("\n" + "="*70)
print("🎯 P1_northwind 用のベンチマーク辞書パラメータ (app.py 組み込み用):")
print("="*70)
print(f'    "mu": np.array([{mu[0]:.2f}, {mu[1]:.2f}]),')
print(f'    "cov": np.array([[{cov[0,0]:.3f}, {cov[0,1]:.3f}], [{cov[1,0]:.3f}, {cov[1,1]:.3f}]]),')
print(f'    "x_range": ({round(mu[0]-10, 1)}, {round(mu[0]+12, 1)}),')
print(f'    "y_range": ({max(25.0, round(mu[1]-25, 1))}, {round(mu[1]+55, 1)}),')
print("="*70)

プロジェクトルート: /Users/kouichinakamura/Documents/GitHub/ninja-phonics
P1 音声確認先: /Users/kouichinakamura/Documents/GitHub/ninja-phonics/benchmarks/P1_northwind
P1 (The North Wind and the Sun) 音声を解析中...
  [US-Native]: 3 ファイル検出
  [UK-Native]: 3 ファイル検出
  [JP-Accented]: 3 ファイル検出
  [JP-Mora-Forced]: 3 ファイル検出

✅ 解析完了: 12 サンプルを保存 -> /Users/kouichinakamura/Documents/GitHub/ninja-phonics/benchmarks/P1_northwind/northwind_benchmark_results.csv

📊 コーホート別 指標サマリー (平均 ± 標準偏差)
                     pct_v   schwa_p75        rcf        npvi
cohort                                                       
JP-Accented     40.7 ± 0.2  61.9 ± 2.6  1.4 ± 0.1  51.4 ± 2.2
JP-Mora-Forced  39.5 ± 2.4  86.0 ± 5.0  1.1 ± 0.1  69.5 ± 3.3
UK-Native       40.1 ± 0.2  60.1 ± 0.0  1.4 ± 0.0  50.5 ± 3.8
US-Native       39.8 ± 1.1  56.0 ± 5.7  1.3 ± 0.1  43.5 ± 1.0

🎯 P1_northwind 用のベンチマーク辞書パラメータ (app.py 組み込み用):
    "mu": np.array([39.94, 58.03]),
    "cov": np.array([[0.835, 4.218], [4.218, 24.163]]),
    "x_range": (29.9, 51.9),